# Reproduce the university-rankings analysis - the "Verify" layer

This is the **reproducible notebook** at the bottom of the Inspector / coding-verifier
layer. It re-runs, *from the bundled data*, the pipeline behind the blog's headline
numbers and **asserts** that the reproduced figures match the published ones - so the
notebook is a *proof*, not just a script.

**Download-and-run-locally.** Download this notebook together with the `verify/data/`
inputs shipped beside it (`core_master.csv` = the 218-institution five-way common core;
`client_model.js` = the reweighter's 218x5 pillar matrix). Run the cells top to bottom.
There is no Colab branch and no remote fetch.

## What it proves
- the five-system common core is **218** universities;
- the four commercial systems correlate only **Spearman 0.60-0.85** (tie-handled);
- the tables crown **four different #1s** (Caltech / Harvard / MIT / Michigan);
- **Peking** swings **89** places (QS 12 vs ARWU 101);
- the neutral OpenAlex referee crowns **Michigan** (works) but ARWU tracks its h-index best (~0.88);
- the reweighter's presets crown **Oxford 87.02 / MIT 96.25 / Michigan 100.0** from the same data.

> Read-only on the bundled data. Regenerated files (if any) go to `verify/_repro_out/`.


In [ ]:
# === Setup: imports, DATA_DIR resolver, guarded reads ===
import os, re, json
from pathlib import Path
import numpy as np, pandas as pd

_CANDIDATES = [
    os.environ.get("UNIRANK_DATA_DIR"),   # explicit override
    "verify/data",                          # inputs shipped beside this notebook
    "data",                                 # cwd = verify/
    "../code", "../../code", "code",        # repo-relative to the analysis code
]
def _resolve(cands):
    for c in cands:
        if not c: continue
        p = Path(c).expanduser().resolve()
        if (p / "core_master.csv").exists():
            return p
    return None
DATA_DIR = _resolve(_CANDIDATES)
assert DATA_DIR is not None, (
    "Could not locate core_master.csv. Download the bundled verify/data/ inputs next to "
    "this notebook, or set UNIRANK_DATA_DIR to the folder that contains core_master.csv.")
for rel in ["core_master.csv", "client_model.js"]:
    assert (DATA_DIR / rel).exists(), f"missing required input: {rel}"

REPRO_OUT = Path("_repro_out").resolve(); REPRO_OUT.mkdir(exist_ok=True)
core = pd.read_csv(DATA_DIR / "core_master.csv")
print("DATA_DIR:", DATA_DIR)
print("core rows:", len(core))


In [ ]:
# === cell_core: the five-way common core is 218 universities (ana_03) ===
n = len(core)
print("five-system common core =", n, "universities")
assert n == 218, f"expected 218 common-core universities, got {n}"
print("OK: matches the published common-core size (218).")


In [ ]:
# === cell_disagreement: correlations, four champions, Peking's swing (ana_04/05b/06) ===
def spearman(a, b):
    return pd.Series(a).rank(method="average").corr(pd.Series(b).rank(method="average"))

pairs = [("the_rank","arwu_rank","THE-ARWU",0.793),
         ("the_rank","cwur_rank","THE-CWUR",0.759),
         ("the_rank","qs_rank","THE-QS",0.735),
         ("arwu_rank","cwur_rank","ARWU-CWUR",0.853),
         ("arwu_rank","qs_rank","ARWU-QS",0.647),
         ("cwur_rank","qs_rank","CWUR-QS",0.599)]
print("pairwise Spearman on the 218-school core:")
for a, b, lab, pub in pairs:
    s = spearman(core[a], core[b])
    print(f"  {lab:10} {s:.3f}  (published {pub:.3f})")
    assert abs(s - pub) < 0.02, f"{lab} Spearman {s:.3f} off published {pub}"
lo = min((spearman(core[a],core[b]),lab) for a,b,lab,_ in pairs)
hi = max((spearman(core[a],core[b]),lab) for a,b,lab,_ in pairs)
print(f"most alike: {hi[1]} | least alike: {lo[1]}")
assert hi[1] == "ARWU-CWUR" and lo[1] == "CWUR-QS"

# four different champions (min rank in each system)
def champ(col): return core.loc[core[col].idxmin(), "the_name"]
oa_champ = core.loc[core["oa_rank_works"].idxmin(), "the_name"]
print("OpenAlex (works) #1 =", oa_champ)
assert "Michigan" in oa_champ

# Peking's cross-system spread = max - min of the four commercial ranks
pk = core[core["the_name"].str.contains("Peking")].iloc[0]
spread = max(pk[["the_rank","arwu_rank","cwur_rank","qs_rank"]]) - min(pk[["the_rank","arwu_rank","cwur_rank","qs_rank"]])
print(f"Peking spread = {int(spread)} (QS {int(pk['qs_rank'])} vs ARWU {int(pk['arwu_rank'])})")
assert int(spread) == 89, f"Peking spread {spread} != 89"
print("OK: correlations, the four different #1s, and Peking's 89-place swing all reproduce.")


In [ ]:
# === cell_referee: the neutral referee reorders everyone (ana_07/08) ===
def spearman(a, b):
    return pd.Series(a).rank(method="average").corr(pd.Series(b).rank(method="average"))
# ARWU tracks OpenAlex h-index most closely; QS least
arwu_h = spearman(core["arwu_rank"], core["oa_rank_hindex"])
qs_h   = spearman(core["qs_rank"],   core["oa_rank_hindex"])
print(f"ARWU vs OpenAlex h-index Spearman = {arwu_h:.3f} (published 0.876)")
print(f"QS   vs OpenAlex h-index Spearman = {qs_h:.3f} (published 0.636)")
assert abs(arwu_h - 0.876) < 0.02 and abs(qs_h - 0.636) < 0.02
assert arwu_h > qs_h, "ARWU should track measured impact more closely than QS"
print("OK: ARWU tracks the referee more closely than QS; the tightest fit still falls short of 1.0.")


In [ ]:
# === cell_client_models: the reweighter's presets crown three different #1s (ana_12) ===
js = (DATA_DIR / "client_model.js").read_text(encoding="utf-8")
m = js.index("const REWEIGHT_DATA"); a = js.index("[", m); b = js.index("];", a) + 1
RW = json.loads(js[a:b])
assert len(RW) == 218, f"reweighter matrix has {len(RW)} rows, expected 218"
KEYS = ["rep","cite","teach","intl","vol"]
def reweight(w):
    tot = sum(w[k] for k in KEYS) or 1
    sc = [(d["name"], round(sum(w[k]*d[k] for k in KEYS)/tot, 2)) for d in RW]
    sc.sort(key=lambda r: -r[1]); return sc
PRESETS = {"balanced":{"rep":1,"cite":1,"teach":1,"intl":1,"vol":1},
           "qs_like":{"rep":5,"cite":2,"teach":2,"intl":1,"vol":0},
           "output_only":{"rep":0,"cite":0,"teach":0,"intl":0,"vol":1}}
expect = {"balanced":("University of Oxford",87.02),
          "qs_like":("Massachusetts Institute of Technology",96.25),
          "output_only":("University of Michigan",100.0)}
for name, w in PRESETS.items():
    top = reweight(w)[0]
    exp_name, exp_score = expect[name]
    print(f"[{name}] #1 {top[0]} ({top[1]})")
    assert top[0] == exp_name and abs(top[1]-exp_score) < 1e-9, f"{name}: got {top}, expected {expect[name]}"
print("OK: balanced -> Oxford 87.02; QS-like -> MIT 96.25; output-only -> Michigan 100.0.")


## Provenance summary & licenses

Every published number above is reproduced from the bundled `verify/data/` inputs.

| Finding (blog) | Reproduced in | Source |
|---|---|---|
| 218-school common core | `cell_core` | `core_master.csv` (from `code/02_alignment.py`) |
| Spearman 0.60-0.85 disagreement | `cell_disagreement` | `core_master.csv` (`code/03_disagreement.py`) |
| Four different #1s; Peking swing 89 | `cell_disagreement` | `core_master.csv` |
| ARWU/QS vs OpenAlex tracking | `cell_referee` | `core_master.csv` (`code/04_referee.py`) |
| Reweighter presets (Oxford/MIT/Michigan) | `cell_client_models` | `client_model.js` (`code/07_client_models.py`) |

### Licenses
- **THE / ARWU / CWUR** editions - arnaudbenard/university-ranking (mirror of the Kaggle "World University Rankings").
- **QS 2023** - GeorgeM2000/QS-World-University-Ranking.
- **OpenAlex** - CC0 open bibliometric index (fetched 24 June 2026).

*Regenerated files land in `verify/_repro_out/`, never back into the read-only data.*
